In [1]:
import torch
from datasets import load_dataset
import numpy as np
from transformers import (
    EncoderDecoderModel,
    EncoderDecoderConfig,
    BertConfig,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
import evaluate

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Split, ByteLevel, Whitespace, Metaspace
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.normalizers import NFKC, Sequence

bleu = evaluate.load("sacrebleu")

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)


In [2]:
tokenizer_si = Tokenizer.from_file("./tokenizers_trained/bpe_bytelevel_si_1m_32k/tokenizer.json")
tokenizer_en = Tokenizer.from_file("./tokenizers_trained/bpe_bytelevel_en_1m_32k/tokenizer.json")

In [3]:
#print special token of tokenizer_si
print("Special tokens in tokenizer_si:")
print(f"PAD_ID: {tokenizer_si.token_to_id('[PAD]')}")
print(f"UNK_ID: {tokenizer_si.token_to_id('[UNK]')}")
print(f"BOS_ID: {tokenizer_si.token_to_id('[BOS]')}")
print(f"EOS_ID: {tokenizer_si.token_to_id('[EOS]')}")
print(f"Vocabulary size si: {tokenizer_si.get_vocab_size()}")

#print special token of tokenizer_en
print("\nSpecial tokens in tokenizer_en:")
print(f"PAD_ID: {tokenizer_en.token_to_id('[PAD]')}")
print(f"UNK_ID: {tokenizer_en.token_to_id('[UNK]')}")
print(f"BOS_ID: {tokenizer_en.token_to_id('[BOS]')}")
print(f"EOS_ID: {tokenizer_en.token_to_id('[EOS]')}")
print(f"Vocabulary size en: {tokenizer_en.get_vocab_size()}")

Special tokens in tokenizer_si:
PAD_ID: 0
UNK_ID: 1
BOS_ID: 2
EOS_ID: 3
Vocabulary size si: 32000

Special tokens in tokenizer_en:
PAD_ID: 0
UNK_ID: 1
BOS_ID: 2
EOS_ID: 3
Vocabulary size en: 32000


In [4]:

PAIR = "en-si"
SRC_LANG = "si"
TGT_LANG = "en"

MAX_LEN = 128
PAD_ID = 0
UNK_ID = 1
BOS_ID = 2
EOS_ID = 3

VOCAB_SIZE_SI = 32000
VOCAB_SIZE_TA = 32000


In [ ]:

# =========================
# MODEL
# =========================

def build_model():

    encoder_config = BertConfig(
        vocab_size=VOCAB_SIZE_SI,
        hidden_size=512,
        num_hidden_layers=6,
        num_attention_heads=8,
        intermediate_size=2048,
        max_position_embeddings=512,
        pad_token_id=PAD_ID,
    )

    decoder_config = BertConfig(
        vocab_size=VOCAB_SIZE_TA,
        hidden_size=512,
        num_hidden_layers=6,
        num_attention_heads=8,
        intermediate_size=2048,
        max_position_embeddings=512,
        is_decoder=True,
        add_cross_attention=True,
        pad_token_id=PAD_ID,
    )

    config = EncoderDecoderConfig.from_encoder_decoder_configs(
        encoder_config, decoder_config
    )

    config.decoder_start_token_id = BOS_ID
    config.bos_token_id = BOS_ID
    config.pad_token_id = PAD_ID
    config.eos_token_id = EOS_ID
    #config.forced_eos_token_id = EOS_ID
    config.max_length = MAX_LEN
    config.num_beams = 4
    config.early_stopping = True

    #check these settings, as they can cause issues if not set correctly.
    config.tie_encoder_decoder = False
    config.tie_word_embeddings = False

    model = EncoderDecoderModel(config)

    return model


In [ ]:

# =========================
# DATASET PREPROCESS
# =========================

def preprocess(example):

    src = example["translation"][SRC_LANG]
    tgt = example["translation"][TGT_LANG]

    src_ids = tokenizer_si.encode(src).ids[:MAX_LEN-2]
    tgt_ids = tokenizer_en.encode(tgt).ids[:MAX_LEN-2]

    src_ids = [BOS_ID] + src_ids + [EOS_ID]
    tgt_ids = [BOS_ID] + tgt_ids + [EOS_ID]

    return {
        "input_ids": src_ids,
        "labels": tgt_ids
    }


def collate_fn(batch):

    input_ids = [torch.tensor(x["input_ids"]) for x in batch]
    labels = [torch.tensor(x["labels"]) for x in batch]

    input_ids = torch.nn.utils.rnn.pad_sequence(
        input_ids, batch_first=True, padding_value=PAD_ID
    )

    labels = torch.nn.utils.rnn.pad_sequence(
        labels, batch_first=True, padding_value=-100    #check if -100 is the correct padding value for labels in seq2seq models
    )

    # Bert models expect decoder_input_ids to be provided during training, so we need to create them from the labels.
    # Create decoder_input_ids (The "Shift Right" fix)
    decoder_input_ids = labels.new_zeros(labels.shape)
    decoder_input_ids[:, 1:] = labels[:, :-1].clone()
    decoder_input_ids[:, 0] = BOS_ID
    decoder_input_ids.masked_fill_(decoder_input_ids == -100, PAD_ID)
    # While optional in some versions, it's best practice to provide this
    decoder_attention_mask = (decoder_input_ids != PAD_ID).long()

    attention_mask = (input_ids != PAD_ID).long()

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "decoder_input_ids": decoder_input_ids,
        "decoder_attention_mask": decoder_attention_mask,
        "labels": labels
    }


def compute_metrics(eval_preds):

    preds, labels = eval_preds

    # In case the model returns more than the prediction logits
    if isinstance(preds, tuple):
        preds = preds[0]

    # Replace -100 in labels so we can decode them
    labels = np.where(labels != -100, labels, PAD_ID)

    decoded_preds = [
        tokenizer_en.decode(p, skip_special_tokens=True)
        for p in preds
    ]

    decoded_labels = [
        tokenizer_en.decode(l, skip_special_tokens=True)
        for l in labels
    ]

    # Some simple post-processing
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    result = bleu.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    return {"bleu": result["score"]}


In [7]:

# =========================
# MAIN
# =========================

dataset = load_dataset("Helsinki-NLP/opus-100", PAIR, cache_dir="./hf_cache")

dataset = dataset.map(preprocess, remove_columns=dataset["train"].column_names)


In [ ]:
# Create a tiny slice of the data
small_train_dataset = dataset["train"].select(range(100))
small_eval_dataset = dataset["validation"].select(range(50))

In [9]:

model = build_model()


In [10]:

training_args = Seq2SeqTrainingArguments(

    output_dir="./checkpoints",
    save_total_limit=2,
    metric_for_best_model="bleu",
    greater_is_better=True,
    load_best_model_at_end=True,

    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,

    num_train_epochs=8,

    learning_rate=3e-4,

    #warmup_steps=1000,
    warmup_ratio = 0.1,

    max_grad_norm=1.0,

    fp16=True,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=100,

    label_smoothing_factor=0.1,

    predict_with_generate=True,

    push_to_hub=False,

    report_to="none"
)

trainer = Seq2SeqTrainer(

    model=model,

    args=training_args,

    #train_dataset=dataset["train"],
    train_dataset=small_train_dataset,

    #eval_dataset=dataset["validation"],
    eval_dataset=small_eval_dataset,

    data_collator=collate_fn,

    compute_metrics=compute_metrics
)


In [11]:
print(f"Is CUDA available? {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

Is CUDA available? True
Device Name: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [13]:

trainer.train()

Epoch,Training Loss,Validation Loss,Bleu
1,No log,9.367517,0.000000
2,No log,8.728702,0.000000
3,No log,8.222311,0.000000
4,No log,7.933120,0.000000
5,No log,7.668528,0.000000
6,No log,7.532537,0.000000
7,No log,7.451564,0.000000
8,No log,7.426417,0.000000


There were missing keys in the checkpoint model loaded: ['decoder.cls.predictions.decoder.weight', 'decoder.cls.predictions.decoder.bias'].


TrainOutput(global_step=32, training_loss=7.412751197814941, metrics={'train_runtime': 13.9293, 'train_samples_per_second': 57.433, 'train_steps_per_second': 2.297, 'total_flos': 12396966174720.0, 'train_loss': 7.412751197814941, 'epoch': 8.0})

In [ ]:
trainer.save_model("./final_translation_model")